# Побудова моделі прийому файлів через повідомлення в EventEtream

## 1. Створення структури таблиці Журналу прийнятих файлів.
---
Таблиця взагаліто створюється автоматично, коли вибираємо LakeHouse як Destiantion. Але краще створити таблицю окремо. Особливо, коли ти додав кілька служблвих полів. Але структура полів таблиці відповіждає структурі повідлмлення з EventStream.

Додані при трансформації поля:

```
- processing_status STRING статус обробки файлу
- processed_at TIMESTAMP дата зміни статусу
```
---


In [1]:
print(f"Create Table dbo.ev2_file_gr")


spark.sql("""

   CREATE TABLE dbo.ev2_file_grn (
  specversion STRING,
  type STRING,
  subject STRING,
  time STRING,
  source STRING,
  id STRING,
  data STRUCT<
    api: STRING,
    clientRequestId: STRING,
    requestId: STRING,
    eTag: STRING,
    contentType: STRING,
    contentLength: BIGINT,
    contentOffset: BIGINT,
    blobUrl: STRING,
    url: STRING,
    sequencer: STRING
  >,
  EventProcessedUtcTime TIMESTAMP,
  PartitionId BIGINT,
  EventEnqueuedUtcTime TIMESTAMP,
  processing_status STRING,
  processed_at TIMESTAMP
)
USING DELTA;     
"""
)

print(f"Table created")

StatementMeta(, 58ac1c4e-6828-4855-835f-f48bc13771d9, 3, Finished, Available, Finished, False)

Create Table EV_FILEGRN FileList
Table created


## 2. Створюю таблицю для тіла прийнятих файлів


In [1]:
print(f"Create Table dbo.ev2_file_body")


spark.sql("""
    CREATE TABLE dbo.ev2_file_body (
            TX_ID         STRING,
            TERMINAL_ID   BIGINT,
            OPDATE        DATE,
            NAME          STRING,
            PASSPORT      STRING,
            AMOUNT        DECIMAL(18,2),
            CHARGE        DECIMAL(18,2),
            STATUS        STRING,
            FILE_NAME     STRING,
            FILE_ID       STRING
    )  USING DELTA;  
"""
)

print(f"Table created")

StatementMeta(, 8d03f244-f1c2-4d61-901a-3bbf4a4a2fa0, 3, Finished, Available, Finished, False)

Create Table dbo.ev2_file_body
Table created


## 3. Створюю таблицю календаря операційних днів

---

По факту, така таблция вже є psh_dwh_wh. Тому я дані просто перенесу з іншої бд


In [ ]:
print(f"Create Table dbo.calendar")


spark.sql("""
    CREATE TABLE psh_dwh_lh.dbo.calendar (
            OPDATE        DATE,
            OPYEAR        BIGINT,
            OPMONTH       INT,
            OPDAYOFWEEK   INT,
            STATUS        STRING

    )  USING DELTA;  
"""
)

print(f"Table created")

### 3.1. Переношу дані з таблциі psh_dwh_wh.payment.CALENDAR_D

Глянути що є в таблиці джерелі

In [ ]:
import pandas as pd
target_table="psh_dwh_lh.dbo.calendar"
source_table="psh_dwh_wh.payment.CALENDAR_D"
df_source=spark.sql( 
    f"""
        SELECT * FROM {source_table}
    """
)
pd_df_sourse = df_source.toPandas()
display(pd_df_source.head(10))


Безпосередньо Перенос даних і побачити зо є в таблці - джерелі

In [ ]:
import pandas as pd
target_table="psh_dwh_lh.dbo.calendar"
source_table="psh_dwh_wh.payment.CALENDAR_D"
spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {source_table} AS source
            ON target.OPDATE = source.OPDATE
            WHEN MATCHED THEN
                UPDATE SET 
                    target.OPDATE = source.OPDATE, 
                    target.OPYEAR = source.OPYEAR,
                    target.OPMONTH  = source.OPMONTH, 
                    target.OPDAYOFWEEK = source.OPDAYOFWEEK,
                    target.STATUS = source.STATUS

            WHEN NOT MATCHED THEN
                INSERT (OPDATE, OPYEAR, OPMONTH, OPDAYOFWEEK, STATUS )
                VALUES (source.OPDATE, source.OPYEAR, source.OPMONTH, source.OPDAYOFWEEK, source.STATUS)
        """)



df_target=spark.sql(f"""SELECT * FROM {target_table}""")
pd_df_target = df_target.toPandas()
#df.show()
display(pd_df_target.head(10))

Подививтися, що є в таблиці джерелі

## 4. Утиліти для роботи та відлагодженя

### 4.1. Виборка  нових файлів  з таблиці журналу файлів  **ev2_file_grn**

In [4]:
import pandas as pd
df=spark.sql("""
   SELECT A.id, A.type, A.subject, A.time, A.processing_status, A.processed_at 
   FROM ev2_file_grn A
   WHERE A.processing_status = 'NEW'
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

StatementMeta(, 0d64b3b3-95c6-4ca4-9bac-ff2a3407dae1, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 67edfa3f-73c6-4c29-a482-a0b7919b03f2)

### 4.2. Виборка  оброблених файлів  з таблиці журналу файлів **ev2_file_grn**

In [2]:
import pandas as pd
df=spark.sql("""
   SELECT A.id, A.type, A.subject, A.time, A.processing_status, A.processed_at 
   FROM ev2_file_grn A
   WHERE A.processing_status = 'PROCESSED'
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

StatementMeta(, 0d64b3b3-95c6-4ca4-9bac-ff2a3407dae1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 911bd13d-67bd-458a-b5a1-9223a493d84c)

### 4.3 Змінити статус обробки файла на NEW (для відладки) **dbo.ev2_file_grn**

In [ ]:
spark.sql("""
            UPDATE ev2_file_grn
            SET 
            processing_status = 'NEW',
            processed_at = current_timestamp()
            WHERE  processing_status = 'PROCESSED'
        """)

### 4.4 Очистка таблиці журналу файлів в процесі відладки **dbo.ev2_file_grn**

In [1]:
print(f"truncate Table dbo.ev2_file_grn")


spark.sql(
"""
TRUNCATE TABLE dbo.ev2_file_grn;
"""
)

print(f"Table trancated")

StatementMeta(, 6892f3f3-aa3f-416b-8029-5905d5cdf1c0, 3, Finished, Available, Finished, False)

truncate Table dbo.ev2_file_grn
Table trancated


## 5. Сервісні функції для роботи з таблицею тіла файлів **ev2_file_body**

#### 5.1. Виборка за талиці  **ev2_file_body**

In [1]:
import pandas as pd
df=spark.sql("""
   SELECT B.*
   FROM ev2_file_body B
   
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

StatementMeta(, df638f3f-28c9-41cd-8376-8c21649c5fa2, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d4ba748-cc30-49d6-a817-810931c6c678)

In [ ]:
import pandas as pd
df=spark.sql("""
   SELECT distinct B.FILE_ID, B.FILE_NAME
   FROM ev2_file_body B
   
"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

#### 5.2. Очистка таблиці  **ev2_file_body**

In [ ]:
print(f"truncate Table dbo.ev2_file_body")


spark.sql(
"""
TRUNCATE TABLE dbo.ev2_file_body;
"""
)

print(f"Table trancated")

#### 5.3. Різні підрахунки для  **ev2_file_body**

In [2]:
import pandas as pd
df=spark.sql("""
   SELECT B.OPDATE , count(*) as trncnt  FROM ev2_file_body B GROUP BY B.OPDATE 

"""
)
pd_df = df.toPandas()
display(pd_df.head(40))

StatementMeta(, 85883932-fec9-4511-b992-5a7db36e6793, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e3b7185b-a207-408c-9c5c-9ee137c94ff2)

## 6. Утілити для копіювання файлів. Для відладки процесу

---

In [3]:
# Копіюю файди в каталог прийому
import json
import os
from notebookutils import mssparkutils


source_folder = "Files/bronze/archive" # Або інша папка поза зоною читання
destination_folder = "Files/terminals"
# 2. Отримання списку JSON
all_files = mssparkutils.fs.ls(source_folder)
json_files = [f.path for f in all_files if f.name.endswith(".json")]

num=3
i=0
print("Копіюю файли")
for file_path in json_files:
    file_name = os.path.basename(file_path)
    destination = os.path.join(destination_folder, file_name)
    
    # Копіюю файли файл
    mssparkutils.fs.cp(file_path, destination)
    i+=1
    if i>=num:
        break


print(f"Всі {len(json_files)} файлів переміщено в {destination_folder}.")


StatementMeta(, 0d64b3b3-95c6-4ca4-9bac-ff2a3407dae1, 9, Finished, Available, Finished, False)

Копіюю файли
Всі 95 файлів переміщено в Files/terminals.
